In [1]:
import pandas as pd

df = pd.read_csv("paket_a_transaksi_pos_2024_2025.csv")

display(df.head())

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


In [2]:
# Memeriksa tipe data dan beberapa nilai harga_satuan
print("Tipe data sebelum diperbaiki:")
print(df["harga_satuan"].dtype)

print("\nContoh nilai harga_satuan:")
print(df["harga_satuan"].head())

# Mengubah tanda koma menjadi titik
df["harga_satuan"] = df["harga_satuan"].astype("string").str.replace(",", ".", regex=False)

# Mengubah kolom menjadi tipe data numerik
df["harga_satuan"] = pd.to_numeric(df["harga_satuan"], errors="coerce").astype(float)

# Memeriksa hasil perubahan
print("\nTipe data setelah diperbaiki:")
print(df["harga_satuan"].dtype)

display(df[["harga_satuan"]].head())

Tipe data sebelum diperbaiki:
str

Contoh nilai harga_satuan:
0      13000
1      21000
2    7500,00
3       6500
4       6000
Name: harga_satuan, dtype: str

Tipe data setelah diperbaiki:
float64


,harga_satuan
0,13000.0
1,21000.0
2,7500.0
3,6500.0
4,6000.0


In [3]:
# Memeriksa jumlah data yang bernilai negatif
print("Jumlah data negatif sebelum diperbaiki:")
print((df["jumlah"] < 0).sum())

# Mengubah nilai jumlah negatif menjadi positif menggunakan .loc[]
df.loc[df["jumlah"] < 0, "jumlah"] = df.loc[df["jumlah"] < 0, "jumlah"].abs()

# Memeriksa kembali jumlah data negatif
print("\nJumlah data negatif setelah diperbaiki:")
print((df["jumlah"] < 0).sum())

Jumlah data negatif sebelum diperbaiki:
41

Jumlah data negatif setelah diperbaiki:
0


In [4]:
# Memeriksa jumlah data kosong pada kode_kasir
print("Jumlah data kosong sebelum diperbaiki:")
print(df["kode_kasir"].isna().sum())

# Mengisi data kosong dengan K_UNKNOWN menggunakan .loc[]
df.loc[df["kode_kasir"].isna(), "kode_kasir"] = "K_UNKNOWN"

# Memeriksa kembali data kosong
print("\nJumlah data kosong setelah diperbaiki:")
print(df["kode_kasir"].isna().sum())

Jumlah data kosong sebelum diperbaiki:
261

Jumlah data kosong setelah diperbaiki:
0


In [5]:
# Menampilkan 5 baris pertama dengan kolom tertentu menggunakan iloc
df.iloc[:5, [df.columns.get_loc(kolom) for kolom in [
    "id_transaksi",
    "tanggal",
    "kode_gerai",
    "jumlah"
]]]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


In [6]:
# Menghitung total kotor setiap transaksi
df["Total_Kotor"] = df["harga_satuan"] * df["jumlah"]

# Menampilkan hasil
display(df[["harga_satuan", "jumlah", "Total_Kotor"]].head())

,harga_satuan,jumlah,Total_Kotor
0,13000.0,4,52000.0
1,21000.0,1,21000.0
2,7500.0,3,22500.0
3,6500.0,2,13000.0
4,6000.0,1,6000.0


In [7]:
# Menghitung nominal diskon setiap transaksi
df["Nominal_Diskon"] = df["Total_Kotor"] * df["diskon_persen"] / 100

# Menampilkan hasil
display(df[["Total_Kotor", "diskon_persen", "Nominal_Diskon"]].head())

,Total_Kotor,diskon_persen,Nominal_Diskon
0,52000.0,0,0.0
1,21000.0,0,0.0
2,22500.0,0,0.0
3,13000.0,0,0.0
4,6000.0,0,0.0


In [8]:
# Menghitung total bersih setelah diskon dan PPN 11%
df["Total_Bersih"] = (df["Total_Kotor"] - df["Nominal_Diskon"]) * 1.11

# Menampilkan hasil perhitungan
display(df[["Total_Kotor", "Nominal_Diskon", "Total_Bersih"]].head())

,Total_Kotor,Nominal_Diskon,Total_Bersih
0,52000.0,0.0,57720.0
1,21000.0,0.0,23310.0
2,22500.0,0.0,24975.0
3,13000.0,0.0,14430.0
4,6000.0,0.0,6660.0


In [9]:
# Memfilter data berdasarkan beberapa kondisi
data_filter = df[
    (df["status"] == "SELESAI") &
    (df["kode_gerai"].isin(["G05", "G09"])) &
    (df["Total_Bersih"] > 100000)
]

# Menampilkan jumlah baris data yang memenuhi kondisi
print("Jumlah transaksi yang memenuhi kondisi:", len(data_filter))

# Menampilkan hasil filter
display(data_filter)

Jumlah transaksi yang memenuhi kondisi: 139


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500.0,10,SELESAI,190000.0,19000.0,189810.0
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,30500.0,0,SELESAI,122000.0,0.0,135420.0
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,28500.0,0,SELESAI,114000.0,0.0,126540.0
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,46500.0,0,SELESAI,139500.0,0.0,154845.0
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,20500.0,0,SELESAI,123000.0,0.0,136530.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4877,TRX0243898,2024-04-23,G05,K008,SGR1204,12,51000.0,0,SELESAI,612000.0,0.0,679320.0
4911,TRX0241966,2024-04-22,G09,K024,SGR1206,3,42000.0,0,SELESAI,126000.0,0.0,139860.0
4925,TRX0240377,2025-06-11,G09,K028,XXX9969,12,51500.0,15,SELESAI,618000.0,92700.0,583083.0
4964,TRX0243219,2025-11-17,G09,K003,SGR1204,6,43500.0,0,SELESAI,261000.0,0.0,289710.0


In [10]:
# Memfilter transaksi gerai G10 dengan jumlah minimal 5
data_query = df.query("kode_gerai == 'G10' and jumlah >= 5")

# Menampilkan hasil filter
display(data_query)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,52000.0,10,SELESAI,312000.0,31200.0,311688.0
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,18000.0,0,SELESAI,108000.0,0.0,119880.0
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,6000.0,0,SELESAI,36000.0,0.0,39960.0
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,54000.0,0,SELESAI,324000.0,0.0,359640.0
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,29000.0,0,SELESAI,174000.0,0.0,193140.0


In [11]:
# Mengurutkan data berdasarkan kode_gerai dan Total_Bersih
df_sorted = df.sort_values(
    by=["kode_gerai", "Total_Bersih"],
    ascending=[True, False]
)

# Menampilkan hasil pengurutan
display(df_sorted)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.00,4.202016e+07
2956,TRX0241367,2025-05-15,G01,K006,RMT1408,12,50500.0,0,SELESAI,606000.0,0.00,6.726600e+05
3588,TRX0240608,2025-07-25,G01,K006,MKN1101,12,47000.0,0,SELESAI,564000.0,0.00,6.260400e+05
2937,TRX0240631,2024-12-24,G01,K013,RMT1408,12,42000.0,0,SELESAI,504000.0,0.00,5.594400e+05
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500.0,10,SELESAI,546000.0,54600.00,5.454540e+05
...,...,...,...,...,...,...,...,...,...,...,...,...
2291,TRX0244124,2024-01-26,G12,K035,MIN1008,1,5500.0,20,SELESAI,5500.0,1100.00,4.884000e+03
3850,TRX0242420,2024-01-29,G12,K019,RMT1406,1,4000.0,0,SELESAI,4000.0,0.00,4.440000e+03
4228,TRX0242364,2025-03-22,G12,K032,PRW1300,1,4500.0,20,SELESAI,4500.0,900.00,3.996000e+03
3919,TRX0244437,2025-09-28,G12,K015,MKN1100,3,16.5,10,SELESAI,49.5,4.95,4.945050e+01


In [12]:
# Mengambil 5 transaksi dengan Total_Bersih terbesar
top_5_transaksi = df.nlargest(5, "Total_Bersih")

# Menampilkan hasil
display(top_5_transaksi)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
994,TRX0241002,2024-03-18,G11,K038,MKN1101,2093,48000.0,10,SELESAI,100464000.0,10046400.0,100363536.0
4985,TRX0244221,2024-12-09,G09,K005,BYI1500,1301,51500.0,10,SELESAI,67001500.0,6700150.0,66934498.5
467,TRX0243696,2024-02-27,G06,K_UNKNOWN,SGR1201,892,54000.0,0,SELESAI,48168000.0,0.0,53466480.0
3393,TRX0241029,2025-05-12,G07,K028,SGR1205,1983,23000.0,10,SELESAI,45609000.0,4560900.0,45563391.0
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0


In [13]:
# Memfilter transaksi yang statusnya SELESAI
df_selesai = df[df["status"] == "SELESAI"]

# Mengelompokkan data berdasarkan kode gerai
laporan_gerai = df_selesai.groupby("kode_gerai").agg(
    Jumlah_Transaksi=("id_transaksi", "count"),
    Total_Unit_Terjual=("jumlah", "sum"),
    Total_Omzet_Bersih=("Total_Bersih", "sum"),
    Rata_Rata_Transaksi=("Total_Bersih", "mean")
)

# Mengurutkan omzet bersih dari terbesar ke terkecil
laporan_gerai = laporan_gerai.sort_values(
    by="Total_Omzet_Bersih",
    ascending=False
).round(2)

# Menampilkan laporan performa gerai
display(laporan_gerai)

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.253536e+08,314958.82
G06,383,3103,9.113936e+07,237961.77
G09,379,3027,9.019606e+07,237984.32
G07,384,4284,8.036181e+07,209275.55
G01,450,2635,7.077959e+07,157287.97
G03,372,3274,5.980970e+07,160778.77
G10,401,1742,5.892248e+07,146938.85
G08,408,1700,3.550256e+07,87016.08
G12,406,1010,2.464483e+07,60701.55


In [14]:
# Menghitung total omzet bersih setiap gerai menggunakan transform
total_omzet_per_gerai = df.groupby("kode_gerai")["Total_Bersih"].transform("sum")

# Menghitung persentase kontribusi setiap transaksi
df["Kontribusi_Gerai_Persen"] = (
    df["Total_Bersih"] / total_omzet_per_gerai
) * 100

# Menampilkan hasil perhitungan
display(df[[
    "kode_gerai",
    "Total_Bersih",
    "Kontribusi_Gerai_Persen"
]].head(10))

,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,G09,57720.0,0.063031
1,G09,23310.0,0.025455
2,G11,24975.0,0.018364
3,G05,14430.0,0.060124
4,G05,6660.0,0.027750
5,G01,545454.0,0.753599
6,G10,19980.0,0.033172
7,G09,189810.0,0.207274
8,G07,36630.0,0.044776
9,G08,14430.0,0.039219
